# EDA

In [10]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sys.path.append(os.path.abspath(".."))
from features.feature_groups import FEATURE_GROUPS

In [11]:
#Plot settings
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
sns.set_style("whitegrid")
palette = sns.color_palette("Set2")

In [12]:
file_path = "../Data/poultry_dataset.parquet"
data = pd.read_parquet(file_path)

In [13]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 74354 entries, 0 to 74353
Data columns (total 65 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   flock_id                   74354 non-null  object 
 1   farm_id                    74354 non-null  object 
 2   house_id                   74354 non-null  object 
 3   cycle_number               74354 non-null  int64  
 4   date                       74354 non-null  object 
 5   age_days                   74354 non-null  int64  
 6   breed                      74354 non-null  object 
 7   placement_count            74354 non-null  int64  
 8   house_type                 74354 non-null  object 
 9   governorate                74354 non-null  object 
 10  stocking_density_per_m2    74354 non-null  float64
 11  house_area_m2              74354 non-null  int64  
 12  doc_quality_score          74354 non-null  float64
 13  doc_arrival_temp_c         74354 non-null  float64
 14  h

In [14]:
print(f"Shape: {data.shape}")
print(f"Unique flocks: {data['flock_id'].nunique()}")
print(f"Unique farms:  {data['farm_id'].nunique()}")
print(f"Unique houses: {data['house_id'].nunique()}")
print(f"Date range:    {data['date'].min()} → {data['date'].max()}")
print(f"Age range:     {data['age_days'].min()} → {data['age_days'].max()} days")

Shape: (74354, 65)
Unique flocks: 1905
Unique farms:  50
Unique houses: 150
Date range:    2023-01-01 → 2024-12-31
Age range:     1 → 42 days


## 1. Missing data analysis

In [15]:
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_pct", ascending=False)
missing_df

,missing_count,missing_pct
antibiotic_name,67640,90.97
disease_event_id,67495,90.78
suspected_disease,67495,90.78
vaccine_given,65299,87.82
vaccine_route,65299,87.82
scheduled_vaccine_today,64829,87.19
co2_ppm,64677,86.99
avg_weight_g,61649,82.91
litter_condition,61019,82.07
ammonia_ppm,42307,56.90


In [17]:
data['antibiotic_name'].unique()

array([None, 'Enrofloxacin', 'Tylosin', 'Colistin', 'Doxycycline',
       'Amoxicillin'], dtype=object)

In [18]:
data['vaccine_route'].unique()

array(['injection', None, 'spray', 'drinking_water'], dtype=object)

### Findings and Thoughts on Handling Missing values
1- Sparse raws: those are not missing but they missured weekly so it's usal.
- Drop: I have to drop these columns disease_event_id, suspected_disease (Ground Truth columns) and use them as validaiton only
  
- Derive features:
- antibiotic_name --> days_since, count_cumulative, window(is it still active for some threshold)
- vaccine_given --> count, window, compliance
- vaccine_route --> flock-level primary route 
- scheduled_vaccine_today --> compliance flag
  
but not sure yet, if every antibioty/vaccine window are different how to handle it.

2-

In [ ]:
# --- Missing data heatmap (visual pattern) ---
cols_with_missing = missing_df.index.tolist()
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(data[cols_with_missing].isnull().T, cbar=False, cmap="YlOrRd", ax=ax)
ax.set_title("Missing Data Pattern (Yellow = Present, Red = Missing)")
ax.set_xlabel("Row index")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# --- Missing % bar chart ---
fig, ax = plt.subplots(figsize=(12, 5))
missing_df["missing_pct"].plot(kind="barh", color=sns.color_palette("OrRd_r", len(missing_df)), ax=ax)
ax.set_title("Missing Data Percentage by Column")
ax.set_xlabel("Missing %")
for i, v in enumerate(missing_df["missing_pct"]):
    ax.text(v + 0.3, i, f"{v}%", va="center", fontsize=9)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 2. TARGET VARIABLE ANALYSIS
# ═══════════════════════════════════════════════════════════════
The main classification target is `mortality_spike` (binary).  
Also investigate `daily_mortality_pct`, `daily_deaths`, `daily_culls`, and cumulative targets.

In [ ]:
# --- 2a. mortality_spike class balance ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
spike_counts = data["mortality_spike"].value_counts()
spike_counts.plot(kind="bar", ax=axes[0], color=[palette[0], palette[1]])
axes[0].set_title("mortality_spike Class Distribution")
axes[0].set_xticklabels(["No Spike (0)", "Spike (1)"], rotation=0)
for i, v in enumerate(spike_counts):
    axes[0].text(i, v + 200, f"{v}\n({v/len(data)*100:.1f}%)", ha="center", fontsize=11)
axes[0].set_ylabel("Count")

# Pie chart
axes[1].pie(spike_counts, labels=["No Spike", "Spike"], autopct="%1.1f%%",
            colors=[palette[0], palette[1]], startangle=90, textprops={"fontsize": 12})
axes[1].set_title("Class Imbalance — Is it heavily skewed?")

plt.suptitle("⚠️ Check: Do we need class balancing techniques (SMOTE, class weights)?", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print(f"\nImbalance ratio: 1:{spike_counts[0]//spike_counts[1]} (majority:minority)")

In [ ]:
# --- 2b. Distribution of daily mortality % ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

data["daily_mortality_pct"].hist(bins=100, ax=axes[0], color=palette[2], edgecolor="black")
axes[0].axvline(0.30, color="red", linestyle="--", label="Spike threshold (0.30%)")
axes[0].set_title("daily_mortality_pct — Full Distribution")
axes[0].set_xlabel("Daily Mortality %")
axes[0].legend()

# Zoomed in on non-zero
data[data["daily_mortality_pct"] > 0]["daily_mortality_pct"].hist(bins=80, ax=axes[1], color=palette[3], edgecolor="black")
axes[1].axvline(0.30, color="red", linestyle="--", label="Spike threshold")
axes[1].set_title("daily_mortality_pct > 0 (Zoomed)")
axes[1].legend()

# Log scale
data[data["daily_mortality_pct"] > 0]["daily_mortality_pct"].hist(bins=80, ax=axes[2], color=palette[4], edgecolor="black")
axes[2].set_yscale("log")
axes[2].axvline(0.30, color="red", linestyle="--", label="Spike threshold")
axes[2].set_title("daily_mortality_pct > 0 (Log Scale)")
axes[2].legend()

plt.suptitle("How is daily mortality distributed? Where does the 0.30% threshold sit?", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2c. Daily deaths & daily culls distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data["daily_deaths"].hist(bins=80, ax=axes[0], color=palette[0], edgecolor="black")
axes[0].set_title("daily_deaths Distribution")
axes[0].set_xlabel("Deaths per day")
axes[0].set_ylabel("Frequency")

data["daily_culls"].hist(bins=80, ax=axes[1], color=palette[1], edgecolor="black")
axes[1].set_title("daily_culls Distribution")
axes[1].set_xlabel("Culls per day")

plt.suptitle("Deaths vs Culls — Are culls underreported? (Check Fridays/Ramadan later)", fontsize=13)
plt.tight_layout()
plt.show()

print("daily_deaths stats:")
print(data["daily_deaths"].describe())
print(f"\nZero-death days: {(data['daily_deaths']==0).sum()} ({(data['daily_deaths']==0).mean()*100:.1f}%)")
print(f"\ndaily_culls stats:")
print(data["daily_culls"].describe())
print(f"\nZero-cull days: {(data['daily_culls']==0).sum()} ({(data['daily_culls']==0).mean()*100:.1f}%)")

In [ ]:
# --- 2d. Cumulative mortality trajectory (sample flocks) ---
sample_flocks = data["flock_id"].drop_duplicates().sample(10, random_state=42)
fig, ax = plt.subplots(figsize=(14, 6))
for flock in sample_flocks:
    flock_data = data[data["flock_id"] == flock].sort_values("age_days")
    ax.plot(flock_data["age_days"], flock_data["cumulative_mortality_pct"], alpha=0.7, label=flock)
ax.set_title("Cumulative Mortality % Over Flock Lifecycle (10 Random Flocks)")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Cumulative Mortality %")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2e. Mortality spike by age_days (when do spikes happen?) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Spike rate by age
spike_by_age = data.groupby("age_days")["mortality_spike"].mean() * 100
spike_by_age.plot(ax=axes[0], color=palette[1])
axes[0].set_title("Mortality Spike Rate by Bird Age")
axes[0].set_xlabel("Age (days)")
axes[0].set_ylabel("Spike Rate (%)")
axes[0].axhline(data["mortality_spike"].mean()*100, color="red", linestyle="--", label="Overall avg")
axes[0].legend()

# Average daily mortality by age
mort_by_age = data.groupby("age_days")["daily_mortality_pct"].mean()
mort_by_age.plot(ax=axes[1], color=palette[2])
axes[1].set_title("Average daily_mortality_pct by Bird Age")
axes[1].set_xlabel("Age (days)")
axes[1].set_ylabel("Avg Daily Mortality %")
axes[1].axhline(0.30, color="red", linestyle="--", label="Spike threshold")
axes[1].legend()

plt.suptitle("At what age are birds most vulnerable?", fontsize=13)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 3. NUMERIC FEATURE DISTRIBUTIONS & DESCRIPTIVE STATS
# ═══════════════════════════════════════════════════════════════
Check for skewness, outliers, and unexpected values.

In [ ]:
# --- 3a. Full descriptive statistics ---
data.describe().T.round(2)

In [ ]:
# --- 3b. Histograms for key numeric features (grid) ---
key_numeric = [
    "placement_count", "stocking_density_per_m2", "house_area_m2",
    "doc_quality_score", "doc_arrival_temp_c", "days_since_feed_delivery",
    "feed_consumed_kg", "feed_per_bird_g", "water_consumed_L",
    "avg_weight_g", "hubbard_standard_weight_g",
    "temp_inside_avg_c", "target_temp_for_age_c", "humidity_inside_pct",
    "ammonia_ppm", "co2_ppm", "fan_runtime_hours", "power_outage_hours",
    "temp_outside_avg_c", "humidity_outside_pct", "wind_speed_kmh",
    "worker_count", "days_since_last_vaccine",
    "prev_cycle_mortality_pct", "prev_cycle_fcr"
]

n_cols = 4
n_rows = (len(key_numeric) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(key_numeric):
    data[col].dropna().hist(bins=50, ax=axes[i], color=palette[i % len(palette)], edgecolor="black", alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distributions of Key Numeric Features", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3c. Boxplots: detect outliers in critical features ---
outlier_cols = [
    "feed_consumed_kg", "feed_per_bird_g", "water_consumed_L",
    "avg_weight_g", "temp_inside_avg_c", "humidity_inside_pct",
    "ammonia_ppm", "power_outage_hours", "wind_speed_kmh",
    "daily_deaths", "daily_culls", "daily_mortality_pct",
    "stocking_density_per_m2", "doc_quality_score", "doc_arrival_temp_c"
]

n_cols = 5
n_rows = (len(outlier_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(outlier_cols):
    data.boxplot(column=col, ax=axes[i])
    axes[i].set_title(col, fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Boxplots — Outlier Detection for Key Features", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3d. Skewness of numeric features ---
numeric_cols = data.select_dtypes(include=[np.number]).columns
skew_df = data[numeric_cols].skew().sort_values(ascending=False).to_frame("skewness")
skew_df["abs_skew"] = skew_df["skewness"].abs()
skew_df = skew_df.sort_values("abs_skew", ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ["red" if abs(s) > 2 else "orange" if abs(s) > 1 else "green" for s in skew_df["skewness"]]
skew_df["skewness"].plot(kind="barh", ax=ax, color=colors)
ax.axvline(0, color="black", linewidth=0.5)
ax.axvline(1, color="orange", linestyle="--", alpha=0.5, label="|skew| > 1 (moderate)")
ax.axvline(-1, color="orange", linestyle="--", alpha=0.5)
ax.axvline(2, color="red", linestyle="--", alpha=0.5, label="|skew| > 2 (high)")
ax.axvline(-2, color="red", linestyle="--", alpha=0.5)
ax.set_title("Feature Skewness — Red = highly skewed (may need transformation)")
ax.legend()
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 4. CATEGORICAL FEATURE DISTRIBUTIONS
# ═══════════════════════════════════════════════════════════════
Check value counts, cardinality, and balance of categorical features.

In [ ]:
# --- 4a. Categorical columns overview ---
cat_cols = data.select_dtypes(include="object").columns.tolist()
cat_summary = pd.DataFrame({
    "nunique": [data[c].nunique() for c in cat_cols],
    "top_value": [data[c].mode().iloc[0] if not data[c].mode().empty else None for c in cat_cols],
    "top_freq": [data[c].value_counts().iloc[0] for c in cat_cols],
    "non_null": [data[c].notna().sum() for c in cat_cols],
    "null_pct": [(data[c].isna().sum() / len(data) * 100).round(1) for c in cat_cols]
}, index=cat_cols)
cat_summary

In [ ]:
# --- 4b. Bar plots for low-cardinality categorical features ---
low_card_cats = ["breed", "house_type", "governorate", "feed_phase", "season",
                 "cooling_pad_status", "litter_condition"]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(low_card_cats):
    vc = data[col].value_counts()
    vc.plot(kind="bar", ax=axes[i], color=palette, edgecolor="black")
    axes[i].set_title(col, fontsize=11)
    axes[i].tick_params(axis="x", rotation=45, labelsize=9)
    for j, v in enumerate(vc):
        axes[i].text(j, v + 50, f"{v/len(data)*100:.1f}%", ha="center", fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Feature Distributions", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 4c. Binary/flag feature distributions ---
binary_cols = ["weighing_done", "khamaseen_dust_storm", "is_holiday", "is_ramadan",
               "antibiotic_given", "vitamin_supplement_given", "mortality_spike"]

fig, axes = plt.subplots(1, len(binary_cols), figsize=(20, 4))
for i, col in enumerate(binary_cols):
    vc = data[col].value_counts()
    vc.plot(kind="bar", ax=axes[i], color=[palette[0], palette[1]], edgecolor="black")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticklabels(["0", "1"], rotation=0)
    axes[i].set_ylabel("")
    total = vc.sum()
    for j, v in enumerate(vc):
        axes[i].text(j, v + 50, f"{v/total*100:.1f}%", ha="center", fontsize=8)

plt.suptitle("Binary/Flag Features — How rare are the events?", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 5. CORRELATION ANALYSIS
# ═══════════════════════════════════════════════════════════════
Find which features are most correlated with the target and with each other (multicollinearity).

In [ ]:
# --- 5a. Full correlation heatmap ---
corr = data.select_dtypes(include=[np.number]).corr()
fig, ax = plt.subplots(figsize=(22, 18))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, annot=False,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1,
            cbar_kws={"shrink": 0.8})
ax.set_title("Full Correlation Heatmap — Look for strong red/blue clusters", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- 5b. Top correlations with mortality_spike ---
target_corr = corr["mortality_spike"].drop(FEATURE_GROUPS["targets"]).sort_values(key=abs, ascending=False)
fig, ax = plt.subplots(figsize=(12, 8))
colors = ["red" if v > 0 else "blue" for v in target_corr]
target_corr.plot(kind="barh", ax=ax, color=colors)
ax.set_title("Feature Correlations with mortality_spike (excluding other targets)")
ax.set_xlabel("Pearson Correlation")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nTop 10 positively correlated:")
print(target_corr.head(10))
print("\nTop 10 negatively correlated:")
print(target_corr.tail(10))

In [ ]:
# --- 5c. Highly correlated feature pairs (multicollinearity check) ---
# Find pairs with |corr| > 0.8 (excluding self-correlations and target group)
high_corr_pairs = []
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            high_corr_pairs.append({
                "Feature_1": corr.columns[i],
                "Feature_2": corr.columns[j],
                "Correlation": round(corr.iloc[i, j], 3)
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values("Correlation", key=abs, ascending=False)
print(f"Feature pairs with |correlation| > 0.8: {len(high_corr_df)}")
print("⚠️ Consider removing or combining these in feature engineering")
high_corr_df

# ═══════════════════════════════════════════════════════════════
# 6. TEMPORAL PATTERNS & SEASONALITY
# ═══════════════════════════════════════════════════════════════
How does mortality behave over time, seasons, months, and special periods?

In [ ]:
# --- 6a. Mortality spike rate by month & season ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# By month
spike_by_month = data.groupby("month")["mortality_spike"].mean() * 100
spike_by_month.plot(kind="bar", ax=axes[0], color=palette, edgecolor="black")
axes[0].set_title("Mortality Spike Rate by Month")
axes[0].set_ylabel("Spike Rate (%)")
axes[0].set_xlabel("Month")
axes[0].axhline(data["mortality_spike"].mean()*100, color="red", linestyle="--", label="Overall avg")
axes[0].legend()

# By season
spike_by_season = data.groupby("season")["mortality_spike"].mean() * 100
spike_by_season.plot(kind="bar", ax=axes[1], color=palette[:4], edgecolor="black")
axes[1].set_title("Mortality Spike Rate by Season")
axes[1].set_ylabel("Spike Rate (%)")
axes[1].set_xlabel("Season")
axes[1].axhline(data["mortality_spike"].mean()*100, color="red", linestyle="--", label="Overall avg")
axes[1].legend()

plt.suptitle("Is there seasonal mortality variation? (Egypt: summer heat is critical)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6b. Daily mortality over calendar time ---
data["date_dt"] = pd.to_datetime(data["date"])
daily_agg = data.groupby("date_dt").agg(
    avg_mortality=("daily_mortality_pct", "mean"),
    spike_rate=("mortality_spike", "mean"),
    total_deaths=("daily_deaths", "sum")
).reset_index()

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True)

axes[0].plot(daily_agg["date_dt"], daily_agg["avg_mortality"], alpha=0.7, linewidth=0.8)
axes[0].set_title("Average Daily Mortality % Over Time")
axes[0].set_ylabel("Avg Mortality %")

axes[1].plot(daily_agg["date_dt"], daily_agg["spike_rate"] * 100, color="red", alpha=0.7, linewidth=0.8)
axes[1].set_title("Mortality Spike Rate Over Time (%)")
axes[1].set_ylabel("Spike Rate %")

axes[2].plot(daily_agg["date_dt"], daily_agg["total_deaths"], color="darkred", alpha=0.7, linewidth=0.8)
axes[2].set_title("Total Deaths Across All Farms Over Time")
axes[2].set_ylabel("Total Deaths")
axes[2].set_xlabel("Date")

plt.suptitle("Time Series View — Any trend, seasonality, or anomalous periods?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6c. Day of week effects (reporting bias check) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

# Spike rate by day of week
spike_dow = data.groupby("day_of_week")["mortality_spike"].mean() * 100
axes[0].bar(dow_labels, spike_dow.values, color=palette, edgecolor="black")
axes[0].set_title("Spike Rate by Day of Week")
axes[0].set_ylabel("Spike Rate (%)")

# Daily culls by day of week — check Friday underreporting
culls_dow = data.groupby("day_of_week")["daily_culls"].mean()
axes[1].bar(dow_labels, culls_dow.values, color=palette, edgecolor="black")
axes[1].set_title("Avg Daily Culls by Day of Week")
axes[1].set_ylabel("Avg Culls")
axes[1].annotate("Friday = less workers?", xy=(4, culls_dow.iloc[4]), fontsize=9, color="red")

# Worker count by day
workers_dow = data.groupby("day_of_week")["worker_count"].mean()
axes[2].bar(dow_labels, workers_dow.values, color=palette, edgecolor="black")
axes[2].set_title("Avg Worker Count by Day of Week")
axes[2].set_ylabel("Workers")

plt.suptitle("Day-of-Week Effects — Friday underreporting of culls?", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 6d. Ramadan & Holiday effect on mortality and operations ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mortality spike rate
for i, (col, title) in enumerate([
    ("is_ramadan", "Ramadan Effect"),
    ("is_holiday", "Holiday Effect")
]):
    grouped = data.groupby(col).agg(
        spike_rate=("mortality_spike", "mean"),
        avg_culls=("daily_culls", "mean"),
        avg_workers=("worker_count", "mean")
    )
    grouped["spike_rate"] *= 100
    grouped["spike_rate"].plot(kind="bar", ax=axes[0] if i == 0 else axes[0],
                                label=title, alpha=0.7)

axes[0].set_title("Spike Rate: Ramadan vs Normal")
data.groupby("is_ramadan")["mortality_spike"].mean().mul(100).plot(kind="bar", ax=axes[0], color=[palette[0], palette[1]])
axes[0].set_xticklabels(["Not Ramadan", "Ramadan"], rotation=0)
axes[0].set_ylabel("Spike Rate %")

data.groupby("is_holiday")["daily_culls"].mean().plot(kind="bar", ax=axes[1], color=[palette[2], palette[3]])
axes[1].set_xticklabels(["Working Day", "Holiday"], rotation=0)
axes[1].set_title("Avg Daily Culls: Holiday vs Working Day")
axes[1].set_ylabel("Avg Culls")

data.groupby("is_ramadan")["worker_count"].mean().plot(kind="bar", ax=axes[2], color=[palette[4], palette[5]])
axes[2].set_xticklabels(["Not Ramadan", "Ramadan"], rotation=0)
axes[2].set_title("Avg Workers: Ramadan Effect")
axes[2].set_ylabel("Workers")

plt.suptitle("Ramadan & Holidays — Impact on Operations and Mortality", fontsize=13)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 7. ENVIRONMENTAL ANALYSIS
# ═══════════════════════════════════════════════════════════════
Temperature, humidity, ammonia, CO2 — key drivers of mortality in poultry.

In [ ]:
# --- 7a. Temperature inside vs target (deviation analysis) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# temp inside vs target scatter
axes[0].scatter(data["target_temp_for_age_c"], data["temp_inside_avg_c"],
                alpha=0.05, s=2, c=data["mortality_spike"], cmap="RdYlGn_r")
axes[0].plot([15, 40], [15, 40], "r--", linewidth=1, label="Perfect match")
axes[0].set_xlabel("Target Temp for Age (°C)")
axes[0].set_ylabel("Actual Inside Temp (°C)")
axes[0].set_title("Inside Temp vs Target — Deviations matter!")
axes[0].legend()

# Temperature deviation distribution
temp_dev = data["temp_inside_avg_c"] - data["target_temp_for_age_c"]
temp_dev.hist(bins=60, ax=axes[1], color=palette[3], edgecolor="black")
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Temp Deviation (Actual - Target)")
axes[1].set_xlabel("Deviation (°C)")

# Deviation vs mortality spike
axes[2].boxplot([temp_dev[data["mortality_spike"] == 0],
                 temp_dev[data["mortality_spike"] == 1]],
                labels=["No Spike", "Spike"])
axes[2].set_title("Temp Deviation by Mortality Spike")
axes[2].set_ylabel("Deviation (°C)")

plt.suptitle("Temperature Control — Is deviation linked to mortality?", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 7b. Inside temp, humidity, ammonia distribution by spike/no-spike ---
env_cols = ["temp_inside_avg_c", "humidity_inside_pct", "ammonia_ppm",
            "temp_outside_avg_c", "power_outage_hours", "wind_speed_kmh"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(env_cols):
    for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
        subset = data[data["mortality_spike"] == spike_val][col].dropna()
        axes[i].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
    axes[i].set_title(f"{col} — by Spike", fontsize=10)
    axes[i].legend(fontsize=8)

plt.suptitle("Environmental Features: Spike vs No-Spike Distribution Comparison", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- 7c. Khamaseen dust storm effect on mortality ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

storm_spike = data.groupby("khamaseen_dust_storm")["mortality_spike"].mean() * 100
storm_spike.plot(kind="bar", ax=axes[0], color=[palette[0], palette[3]], edgecolor="black")
axes[0].set_xticklabels(["No Storm", "Khamaseen Storm"], rotation=0)
axes[0].set_title("Mortality Spike Rate During Khamaseen")
axes[0].set_ylabel("Spike Rate %")

storm_deaths = data.groupby("khamaseen_dust_storm")["daily_deaths"].mean()
storm_deaths.plot(kind="bar", ax=axes[1], color=[palette[0], palette[3]], edgecolor="black")
axes[1].set_xticklabels(["No Storm", "Khamaseen Storm"], rotation=0)
axes[1].set_title("Avg Daily Deaths During Khamaseen")
axes[1].set_ylabel("Avg Deaths")

plt.suptitle("Khamaseen Dust Storm Impact — A known killer in Egyptian poultry", fontsize=13)
plt.tight_layout()
plt.show()
print(f"Khamaseen days: {data['khamaseen_dust_storm'].sum()} ({data['khamaseen_dust_storm'].mean()*100:.1f}% of data)")

In [ ]:
# --- 7d. Temperature range (max - min) as stress indicator ---
data["temp_range_c"] = data["temp_inside_max_c"] - data["temp_inside_min_c"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data["temp_range_c"].hist(bins=50, ax=axes[0], color=palette[5], edgecolor="black")
axes[0].set_title("Temp Range (Max - Min) Distribution")
axes[0].set_xlabel("Temperature Range (°C)")

axes[1].boxplot([data[data["mortality_spike"] == 0]["temp_range_c"],
                 data[data["mortality_spike"] == 1]["temp_range_c"]],
                labels=["No Spike", "Spike"])
axes[1].set_title("Temp Range by Mortality Spike")
axes[1].set_ylabel("Temp Range (°C)")

plt.suptitle("Large daily temperature swings = thermal stress for birds", fontsize=13)
plt.tight_layout()
plt.show()

# cleanup temp column
data.drop(columns=["temp_range_c"], inplace=True)

# ═══════════════════════════════════════════════════════════════
# 8. FARM, BREED & HOUSE ANALYSIS
# ═══════════════════════════════════════════════════════════════
Are some farms/breeds/house types consistently worse?

In [ ]:
# --- 8a. Mortality spike rate by breed ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

breed_spike = data.groupby("breed")["mortality_spike"].mean() * 100
breed_spike.plot(kind="bar", ax=axes[0], color=palette[:3], edgecolor="black")
axes[0].set_title("Spike Rate by Breed")
axes[0].set_ylabel("Spike Rate %")
axes[0].tick_params(axis="x", rotation=0)

# Mortality by house type
house_spike = data.groupby("house_type")["mortality_spike"].mean() * 100
house_spike.plot(kind="bar", ax=axes[1], color=palette[3:6], edgecolor="black")
axes[1].set_title("Spike Rate by House Type")
axes[1].set_ylabel("Spike Rate %")
axes[1].tick_params(axis="x", rotation=0)

# Avg daily mortality by breed
breed_mort = data.groupby("breed")["daily_mortality_pct"].mean()
breed_mort.plot(kind="bar", ax=axes[2], color=palette[:3], edgecolor="black")
axes[2].set_title("Avg Daily Mortality % by Breed")
axes[2].set_ylabel("Avg Mortality %")
axes[2].tick_params(axis="x", rotation=0)

plt.suptitle("Breed & House Type — Structural Differences in Mortality", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 8b. Farm-level mortality variation ---
farm_mort = data.groupby("farm_id").agg(
    spike_rate=("mortality_spike", "mean"),
    avg_daily_mort=("daily_mortality_pct", "mean"),
    total_flocks=("flock_id", "nunique"),
    total_days=("flock_id", "count")
).sort_values("spike_rate", ascending=False)
farm_mort["spike_rate"] *= 100

fig, axes = plt.subplots(2, 1, figsize=(18, 10))

farm_mort["spike_rate"].plot(kind="bar", ax=axes[0], color=palette[1], edgecolor="black")
axes[0].set_title("Mortality Spike Rate by Farm (sorted)")
axes[0].set_ylabel("Spike Rate %")
axes[0].axhline(farm_mort["spike_rate"].mean(), color="red", linestyle="--", label="Mean")
axes[0].legend()

farm_mort["avg_daily_mort"].plot(kind="bar", ax=axes[1], color=palette[2], edgecolor="black")
axes[1].set_title("Average Daily Mortality % by Farm")
axes[1].set_ylabel("Avg Mortality %")
axes[1].axhline(farm_mort["avg_daily_mort"].mean(), color="red", linestyle="--", label="Mean")
axes[1].legend()

plt.suptitle("Farm-Level Variation — Are some farms problematic?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(f"\nTop 5 worst farms by spike rate:")
print(farm_mort.head())

In [ ]:
# --- 8c. DOC quality score vs mortality ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# DOC quality distribution
data["doc_quality_score"].hist(bins=30, ax=axes[0], color=palette[4], edgecolor="black")
axes[0].set_title("DOC Quality Score Distribution")
axes[0].set_xlabel("Quality Score (1-10)")

# DOC quality vs spike (violin)
for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
    subset = data[data["mortality_spike"] == spike_val]["doc_quality_score"]
    axes[1].hist(subset, bins=30, alpha=0.5, color=color, label=label, density=True)
axes[1].set_title("DOC Quality by Mortality Spike")
axes[1].legend()

# DOC arrival temp vs spike
for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
    subset = data[data["mortality_spike"] == spike_val]["doc_arrival_temp_c"]
    axes[2].hist(subset, bins=30, alpha=0.5, color=color, label=label, density=True)
axes[2].set_title("DOC Arrival Temp by Mortality Spike")
axes[2].set_xlabel("Arrival Temp (°C)")
axes[2].legend()

plt.suptitle("Day-Old Chick Quality & Transport — Early Life Stress Impact", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 8d. Hatchery quality comparison ---
hatchery_mort = data.groupby("hatchery_id").agg(
    spike_rate=("mortality_spike", "mean"),
    avg_doc_quality=("doc_quality_score", "mean"),
    flock_count=("flock_id", "nunique")
).sort_values("spike_rate", ascending=False)
hatchery_mort["spike_rate"] *= 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

hatchery_mort["spike_rate"].plot(kind="bar", ax=axes[0], color=palette[3], edgecolor="black")
axes[0].set_title("Spike Rate by Hatchery")
axes[0].set_ylabel("Spike Rate %")

hatchery_mort["avg_doc_quality"].plot(kind="bar", ax=axes[1], color=palette[4], edgecolor="black")
axes[1].set_title("Avg DOC Quality Score by Hatchery")
axes[1].set_ylabel("Quality Score")

plt.suptitle("Hatchery Supply Chain — Quality Differences?", fontsize=13)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 9. FEED, WATER & WEIGHT ANALYSIS
# ═══════════════════════════════════════════════════════════════
Feed and water consumption patterns, weight growth curves.

In [ ]:
# --- 9a. Weight growth curve vs Hubbard standard ---
weight_data = data[data["weighing_done"] == 1].copy()
weight_by_age = weight_data.groupby("age_days").agg(
    avg_weight=("avg_weight_g", "mean"),
    std_weight=("avg_weight_g", "std"),
    standard=("hubbard_standard_weight_g", "mean")
)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(weight_by_age.index, weight_by_age["avg_weight"], "b-", linewidth=2, label="Actual Avg Weight")
ax.fill_between(weight_by_age.index,
                weight_by_age["avg_weight"] - weight_by_age["std_weight"],
                weight_by_age["avg_weight"] + weight_by_age["std_weight"],
                alpha=0.2, color="blue", label="±1 Std Dev")
ax.plot(weight_by_age.index, weight_by_age["standard"], "r--", linewidth=2, label="Hubbard Standard")
ax.set_title("Flock Weight Growth Curve vs Breed Standard")
ax.set_xlabel("Age (days)")
ax.set_ylabel("Weight (g)")
ax.legend()
plt.tight_layout()
plt.show()

# Note: ~1% data entry errors in avg_weight_g per data dictionary
print(f"Weighing days in dataset: {len(weight_data)} ({len(weight_data)/len(data)*100:.1f}%)")
print(f"Weight range: {weight_data['avg_weight_g'].min():.0f}g — {weight_data['avg_weight_g'].max():.0f}g")

In [ ]:
# --- 9b. Feed per bird and water consumed by age ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feed per bird by age
feed_age = data.groupby("age_days")["feed_per_bird_g"].mean()
feed_age.plot(ax=axes[0], color=palette[0])
axes[0].set_title("Avg Feed per Bird by Age")
axes[0].set_xlabel("Age (days)")
axes[0].set_ylabel("Feed per bird (g)")

# Water by age
water_age = data.groupby("age_days")["water_consumed_L"].mean()
water_age.plot(ax=axes[1], color=palette[2])
axes[1].set_title("Avg Water Consumed by Age")
axes[1].set_xlabel("Age (days)")
axes[1].set_ylabel("Water (L)")

# Feed phase transitions
feed_phase_age = data.groupby(["age_days", "feed_phase"]).size().unstack(fill_value=0)
feed_phase_age.plot(kind="area", stacked=True, ax=axes[2], alpha=0.7, colormap="Set2")
axes[2].set_title("Feed Phase by Age (When do transitions happen?)")
axes[2].set_xlabel("Age (days)")
axes[2].set_ylabel("Count")

plt.suptitle("Feed & Water Patterns Over Flock Lifecycle", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 9c. Water-to-feed ratio insight (not creating feature, just EDA) ---
wf_data = data.dropna(subset=["water_consumed_L", "feed_consumed_kg"]).copy()
wf_data["water_feed_ratio"] = wf_data["water_consumed_L"] / wf_data["feed_consumed_kg"].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wf_data["water_feed_ratio"].hist(bins=60, ax=axes[0], color=palette[5], edgecolor="black")
axes[0].set_title("Water-to-Feed Ratio Distribution")
axes[0].set_xlabel("Ratio (L/kg)")
axes[0].axvline(wf_data["water_feed_ratio"].median(), color="red", linestyle="--", label="Median")
axes[0].legend()

# By spike
for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
    subset = wf_data[wf_data["mortality_spike"] == spike_val]["water_feed_ratio"].dropna()
    axes[1].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
axes[1].set_title("Water-to-Feed Ratio by Mortality Spike")
axes[1].set_xlabel("Ratio (L/kg)")
axes[1].legend()

plt.suptitle("Water/Feed Ratio — Drops before outbreaks (per domain knowledge)", fontsize=13)
plt.tight_layout()
plt.show()
print(f"💡 Recommendation: Create water_feed_ratio feature in feature engineering")

# ═══════════════════════════════════════════════════════════════
# 10. HEALTH EVENTS & VACCINATION ANALYSIS
# ═══════════════════════════════════════════════════════════════
Vaccines, antibiotics, supplements — and their relationship to mortality.

In [ ]:
# --- 10a. Vaccine analysis ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Which vaccines are given most?
vaccine_counts = data["vaccine_given"].dropna().value_counts()
vaccine_counts.plot(kind="barh", ax=axes[0, 0], color=palette, edgecolor="black")
axes[0, 0].set_title("Vaccine Types Given (count)")
axes[0, 0].set_xlabel("Count")

# Vaccine routes
route_counts = data["vaccine_route"].dropna().value_counts()
route_counts.plot(kind="bar", ax=axes[0, 1], color=palette, edgecolor="black")
axes[0, 1].set_title("Vaccine Route Distribution")
axes[0, 1].tick_params(axis="x", rotation=45)

# Spike rate around vaccination days
data["vaccine_day"] = data["vaccine_given"].notna().astype(int)
vax_spike = data.groupby("vaccine_day")["mortality_spike"].mean() * 100
vax_spike.plot(kind="bar", ax=axes[1, 0], color=[palette[0], palette[1]], edgecolor="black")
axes[1, 0].set_xticklabels(["No Vaccine", "Vaccine Given"], rotation=0)
axes[1, 0].set_title("Spike Rate on Vaccine Days vs Non-Vaccine Days")
axes[1, 0].set_ylabel("Spike Rate %")

# Scheduled vs actual vaccine compliance
scheduled = data["scheduled_vaccine_today"].notna().sum()
given = data["vaccine_given"].notna().sum()
axes[1, 1].bar(["Scheduled", "Actually Given"], [scheduled, given], color=[palette[2], palette[3]], edgecolor="black")
axes[1, 1].set_title("Vaccine Compliance: Scheduled vs Given")
axes[1, 1].set_ylabel("Count")
for i, v in enumerate([scheduled, given]):
    axes[1, 1].text(i, v + 50, str(v), ha="center")

plt.suptitle("Vaccination Patterns — Compliance and stress response", fontsize=14)
plt.tight_layout()
plt.show()

data.drop(columns=["vaccine_day"], inplace=True)

In [ ]:
# --- 10b. Antibiotic usage and mortality ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Antibiotic usage rate
abx_rate = data["antibiotic_given"].value_counts()
abx_rate.plot(kind="bar", ax=axes[0], color=[palette[0], palette[1]], edgecolor="black")
axes[0].set_xticklabels(["No Antibiotic", "Antibiotic Given"], rotation=0)
axes[0].set_title("Antibiotic Usage")
for i, v in enumerate(abx_rate):
    axes[0].text(i, v + 200, f"{v/len(data)*100:.1f}%", ha="center")

# Which antibiotics?
abx_names = data["antibiotic_name"].dropna().value_counts()
abx_names.plot(kind="barh", ax=axes[1], color=palette, edgecolor="black")
axes[1].set_title("Antibiotic Types Used")

# Spike rate with antibiotics
abx_spike = data.groupby("antibiotic_given")["mortality_spike"].mean() * 100
abx_spike.plot(kind="bar", ax=axes[2], color=[palette[2], palette[3]], edgecolor="black")
axes[2].set_xticklabels(["No Antibiotic", "Antibiotic Given"], rotation=0)
axes[2].set_title("Spike Rate: Antibiotic vs No Antibiotic")
axes[2].set_ylabel("Spike Rate %")

plt.suptitle("Antibiotic Usage — Are antibiotics given reactively (after spikes start)?", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# --- 10c. Litter condition and mortality ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

litter_data = data.dropna(subset=["litter_condition"])
litter_spike = litter_data.groupby("litter_condition")["mortality_spike"].mean() * 100
litter_spike.plot(kind="bar", ax=axes[0], color=palette[:3], edgecolor="black")
axes[0].set_title("Spike Rate by Litter Condition")
axes[0].set_ylabel("Spike Rate %")
axes[0].tick_params(axis="x", rotation=0)

litter_mort = litter_data.groupby("litter_condition")["daily_mortality_pct"].mean()
litter_mort.plot(kind="bar", ax=axes[1], color=palette[:3], edgecolor="black")
axes[1].set_title("Avg Mortality % by Litter Condition")
axes[1].set_ylabel("Avg Mortality %")
axes[1].tick_params(axis="x", rotation=0)

plt.suptitle("Litter Condition — Wet litter = ammonia = disease risk", fontsize=13)
plt.tight_layout()
plt.show()
print(f"💡 Litter recorded weekly only. Fill forward within each flock during feature engineering.")

# ═══════════════════════════════════════════════════════════════
# 11. DISEASE GROUND TRUTH (Validation Only — NOT for modeling)
# ═══════════════════════════════════════════════════════════════
These columns reveal simulated outbreak truth. Use only for understanding, never as features.

In [ ]:
# --- 11a. Disease event overview ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# How many days have active disease events?
disease_counts = data["disease_event_id"].notna().value_counts()
axes[0].pie(disease_counts, labels=["No Disease", "Active Disease"],
            autopct="%1.1f%%", colors=[palette[0], palette[3]])
axes[0].set_title("Days with Active Disease Events")

# Disease types
disease_types = data["suspected_disease"].dropna().value_counts()
disease_types.plot(kind="bar", ax=axes[1], color=palette[1:4], edgecolor="black")
axes[1].set_title("Suspected Disease Distribution")
axes[1].tick_params(axis="x", rotation=0)

# Spike rate by disease type
disease_spike = data.groupby(data["suspected_disease"].fillna("None"))["mortality_spike"].mean() * 100
disease_spike.sort_values(ascending=False).plot(kind="bar", ax=axes[2], color=palette, edgecolor="black")
axes[2].set_title("Spike Rate by Disease Type")
axes[2].set_ylabel("Spike Rate %")
axes[2].tick_params(axis="x", rotation=0)

plt.suptitle("⚠️ Disease Ground Truth — For UNDERSTANDING only, NOT model features!", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nDisease events: {data['disease_event_id'].notna().sum()} days ({data['disease_event_id'].notna().mean()*100:.1f}%)")
print(f"Unique disease events: {data['disease_event_id'].nunique()}")

In [ ]:
# --- 11b. Mortality pattern during disease events (sample flocks) ---
disease_flocks = data[data["disease_event_id"].notna()]["flock_id"].unique()
sample_disease_flocks = np.random.RandomState(42).choice(disease_flocks, min(6, len(disease_flocks)), replace=False)

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for i, flock in enumerate(sample_disease_flocks):
    flock_data = data[data["flock_id"] == flock].sort_values("age_days")
    ax = axes[i]
    ax.plot(flock_data["age_days"], flock_data["daily_mortality_pct"], "b-", alpha=0.7)
    
    # Shade disease windows
    disease_days = flock_data[flock_data["disease_event_id"].notna()]
    if not disease_days.empty:
        ax.axvspan(disease_days["age_days"].min(), disease_days["age_days"].max(),
                   alpha=0.2, color="red", label=disease_days["suspected_disease"].iloc[0])
    
    ax.set_title(flock, fontsize=9)
    ax.set_xlabel("Age (days)")
    ax.set_ylabel("Daily Mortality %")
    ax.legend(fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Disease Event Windows (Red) and Mortality Patterns", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 12. HISTORICAL FEATURES & STOCKING DENSITY
# ═══════════════════════════════════════════════════════════════
Previous cycle performance and house utilization.

In [ ]:
# --- 12a. Historical features ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Previous cycle mortality
data["prev_cycle_mortality_pct"].dropna().hist(bins=50, ax=axes[0], color=palette[0], edgecolor="black")
axes[0].set_title("Previous Cycle Mortality % Distribution")
axes[0].set_xlabel("Prev Cycle Mortality %")

# Previous cycle FCR
data["prev_cycle_fcr"].dropna().hist(bins=50, ax=axes[1], color=palette[1], edgecolor="black")
axes[1].set_title("Previous Cycle FCR Distribution")
axes[1].set_xlabel("Prev Cycle FCR")

# Previous cycle mortality vs current spike
prev_mort_data = data.dropna(subset=["prev_cycle_mortality_pct"])
for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
    subset = prev_mort_data[prev_mort_data["mortality_spike"] == spike_val]["prev_cycle_mortality_pct"]
    axes[2].hist(subset, bins=50, alpha=0.5, color=color, label=label, density=True)
axes[2].set_title("Prev Cycle Mortality by Current Spike")
axes[2].legend()

plt.suptitle("Historical Performance — Does the house's past predict current risk?", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Missing prev_cycle_mortality_pct: {data['prev_cycle_mortality_pct'].isna().sum()} "
      f"({data['prev_cycle_mortality_pct'].isna().mean()*100:.1f}%) — expected for first cycle of each house")

In [ ]:
# --- 12b. Stocking density and house cycles ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stocking density vs spike
for spike_val, color, label in [(0, palette[0], "No Spike"), (1, palette[1], "Spike")]:
    subset = data[data["mortality_spike"] == spike_val]["stocking_density_per_m2"]
    axes[0].hist(subset, bins=40, alpha=0.5, color=color, label=label, density=True)
axes[0].set_title("Stocking Density by Spike")
axes[0].set_xlabel("Birds per m²")
axes[0].legend()

# House total cycles vs spike
cycle_spike = data.groupby("house_total_cycles")["mortality_spike"].mean() * 100
cycle_spike.plot(ax=axes[1], marker="o", color=palette[2])
axes[1].set_title("Spike Rate by House Total Cycles")
axes[1].set_xlabel("Total Cycles Run")
axes[1].set_ylabel("Spike Rate %")

# Placement count distribution
data["placement_count"].hist(bins=40, ax=axes[2], color=palette[3], edgecolor="black")
axes[2].set_title("Placement Count Distribution")
axes[2].set_xlabel("Birds Placed")

plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 13. KEY PAIRWISE RELATIONSHIPS WITH TARGET
# ═══════════════════════════════════════════════════════════════
Deep dive into how the most promising features relate to mortality_spike.

In [ ]:
# --- 13a. Key features vs mortality_spike (violin/box plots) ---
important_features = [
    "feed_per_bird_g", "water_consumed_L", "temp_inside_avg_c",
    "humidity_inside_pct", "ammonia_ppm", "power_outage_hours",
    "worker_count", "days_since_last_vaccine",
    "doc_quality_score", "stocking_density_per_m2"
]

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
axes = axes.flatten()

for i, col in enumerate(important_features):
    col_data = data[[col, "mortality_spike"]].dropna()
    parts = [col_data[col_data["mortality_spike"] == 0][col],
             col_data[col_data["mortality_spike"] == 1][col]]
    bp = axes[i].boxplot(parts, labels=["No Spike", "Spike"], patch_artist=True)
    bp["boxes"][0].set_facecolor(palette[0])
    bp["boxes"][1].set_facecolor(palette[1])
    axes[i].set_title(col, fontsize=10)

plt.suptitle("Feature Distributions: Spike vs No-Spike — Which features separate the classes?", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 13b. Scatter matrix of top features vs daily_mortality_pct ---
scatter_cols = ["daily_mortality_pct", "feed_per_bird_g", "water_consumed_L",
                "temp_inside_avg_c", "ammonia_ppm", "age_days"]

sample = data[scatter_cols].dropna().sample(5000, random_state=42)
pd.plotting.scatter_matrix(sample, figsize=(16, 16), alpha=0.1, diagonal="hist",
                           hist_kwds={"bins": 40, "edgecolor": "black"})
plt.suptitle("Pairwise Scatter Matrix — Top Features", fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════
# 14. EDA SUMMARY & RECOMMENDATIONS FOR NEXT PHASE
# ═══════════════════════════════════════════════════════════════

## Key Observations to Investigate Above:

### Target Variable
- **Class imbalance**: Check how skewed `mortality_spike` is — may need SMOTE, class weights, or threshold tuning
- **Age vulnerability**: Which age ranges see the most spikes?
- **Cumulative mortality curves**: Do they follow expected patterns?

### Missing Data  
- `co2_ppm` (~87% missing) — tunnel houses only, consider dropping or creating a flag
- `ammonia_ppm` (~57% missing) — sensor-dependent, impute or flag
- `fan_runtime_hours` / `cooling_pad_status` (~29% missing) — open-sided houses, handle by house type
- `avg_weight_g` (~83% missing) — by design (weekly weighing), forward-fill or interpolate
- `litter_condition` (~82% missing) — weekly recording, forward-fill within flock

### Feature Engineering Ideas (for next phase)
1. `temp_deviation = temp_inside_avg_c - target_temp_for_age_c`
2. `weight_deviation = avg_weight_g - hubbard_standard_weight_g`  
3. `water_feed_ratio = water_consumed_L / feed_consumed_kg`
4. `temp_range = temp_inside_max_c - temp_inside_min_c`
5. **Lag features**: mortality lag1, lag2, lag3, rolling mean 3/5/7 days
6. `cumulative_fcr = cumulative_feed / (avg_weight * flock_size)`
7. `days_since_last_antibiotic` from `antibiotic_given`
8. Age-phase flags from `age_days`

### Data Quality
- Check `avg_weight_g` for ~1% data entry errors (per data dictionary)
- `daily_culls` may be underreported on Fridays/Ramadan
- `disease_event_id` and `suspected_disease` — NEVER use as features

### Reporting Bias
- Friday and holiday effects on culling data
- Ramadan impact on worker counts and operations